In [2]:
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

# Import
from rag.hybrid_retriever import HybridRetriever

# Initialize
print("\nInitializing Hybrid Retriever...")

hybrid = HybridRetriever()

print("\nHybrid Retriever initialized successfully!")
print(hybrid.__dict__.keys())

Project root:
C:\Users\User\RAG SYS\Question_Generation

Initializing Hybrid Retriever...
INITIALIZING HYBRID RETRIEVER

Loading dense retriever...
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384
Loading FAISS index...
Loading metadata...

RETRIEVER INITIALIZED
Vectors: 2150
Metadata: 2150
Dimension: 384
Top-K: 20
Dense retriever ready.

Loading BM25 retriever...
INITIALIZING BM25

Loading question data...
Records loaded: 2150
Questions extracted: 2150

Building BM25 index...
BM25 index built successfully.

BM25 READY
BM25 retriever ready.

HYBRID RETRIEVER READY
Dense top-k : 20
BM25 top-k  : 20
RRF k       : 60

Hybrid Retriever initialized successfully!
dict_keys(['dense_top_k', 'bm25_top_k', 'rrf_k', 'dense_retriever', 'bm25_retriever'])


In [3]:
# ============================================================
# HYBRID RETRIEVAL — PARAPHRASE EVALUATION
# ============================================================

import json
from pathlib import Path

EVAL_FILE = Path(
    r"C:\Users\User\RAG SYS\Question_Generation\tests\retrieval_eval.json"
)

with open(EVAL_FILE, "r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

print("=" * 70)
print("HYBRID PARAPHRASE RETRIEVAL EVALUATION")
print("=" * 70)

total = len(evaluation_data)

print(f"\nEvaluation queries: {total}")


# ============================================================
# METRICS
# ============================================================

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0

reciprocal_ranks = []

failed = []


# ============================================================
# RUN EVALUATION
# ============================================================

for i, item in enumerate(evaluation_data, 1):

    query = item["query"]
    expected_id = item["expected_id"]

    results = hybrid.retrieve(
        query,
        top_k=5
    )

    retrieved_ids = [
        result["id"]
        for result in results
    ]

    # --------------------------------------------------------
    # Recall@1
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:1]:
        recall_at_1 += 1

    # --------------------------------------------------------
    # Recall@3
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:3]:
        recall_at_3 += 1

    # --------------------------------------------------------
    # Recall@5
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:5]:
        recall_at_5 += 1

    # --------------------------------------------------------
    # MRR
    # --------------------------------------------------------

    if expected_id in retrieved_ids:

        rank = retrieved_ids.index(expected_id) + 1
        reciprocal_ranks.append(1 / rank)

    else:

        reciprocal_ranks.append(0)

        failed.append({
            "query": query,
            "expected_id": expected_id,
            "retrieved_ids": retrieved_ids
        })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if i % 20 == 0 or i == total:
        print(
            f"Processed {i}/{total} "
            f"({i / total * 100:.1f}%)"
        )


# ============================================================
# CALCULATE RESULTS
# ============================================================

recall_1 = recall_at_1 / total
recall_3 = recall_at_3 / total
recall_5 = recall_at_5 / total

mrr = sum(reciprocal_ranks) / total


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("HYBRID PARAPHRASE RESULTS")
print("=" * 70)

print(f"\nTotal queries: {total}")

print(f"Recall@1: {recall_1 * 100:.2f}%")
print(f"Recall@3: {recall_3 * 100:.2f}%")
print(f"Recall@5: {recall_5 * 100:.2f}%")
print(f"MRR:      {mrr:.4f}")

print(f"\nFailed @5: {len(failed)}")

print("\n" + "=" * 70)

HYBRID PARAPHRASE RETRIEVAL EVALUATION

Evaluation queries: 200
Processed 20/200 (10.0%)
Processed 40/200 (20.0%)
Processed 60/200 (30.0%)
Processed 80/200 (40.0%)
Processed 100/200 (50.0%)
Processed 120/200 (60.0%)
Processed 140/200 (70.0%)
Processed 160/200 (80.0%)
Processed 180/200 (90.0%)
Processed 200/200 (100.0%)


HYBRID PARAPHRASE RESULTS

Total queries: 200
Recall@1: 96.50%
Recall@3: 100.00%
Recall@5: 100.00%
MRR:      0.9825

Failed @5: 0



In [4]:
query = "Explain object detection in computer vision."

results = hybrid.retrieve(
    query,
    top_k=5
)

print("=" * 70)
print("HYBRID TEST")
print("=" * 70)

print("Query:", query)

for rank, result in enumerate(results, 1):

    metadata = result["metadata"]["metadata"]

    print(
        f"\n{rank}. "
        f"[RRF: {result['score']:.6f}] "
        f"{result['id']}"
    )

    print("Question:", metadata["question"])

HYBRID TEST
Query: Explain object detection in computer vision.

1. [RRF: 0.031778] ai_ml_computer_vision_q021
Question: What is object detection?

2. [RRF: 0.031010] ai_ml_computer_vision_q023
Question: What is a bounding box in object detection?

3. [RRF: 0.029199] ai_ml_computer_vision_q042
Question: What is the difference between face detection and face recognition?

4. [RRF: 0.028787] ai_ml_computer_vision_q022
Question: What is the difference between image classification and object detection?

5. [RRF: 0.028382] ai_ml_computer_vision_q002
Question: What are common applications of Computer Vision?
